# Day 049 — Exercise 3: train_classifier

**What you'll build:** `train_classifier(X_train, y_train, max_iter=1000) -> LogisticRegression` — train a logistic regression model for binary classification.

**Why it matters:** Logistic Regression is to classification what Linear Regression is to regression — the fundamental linear model you use first before trying something more complex. It learns the decision boundary that best separates classes, and its coefficients are directly interpretable as log-odds per unit change in a feature.

## Provided: Setup + cross_validate_model + overfitting_report

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                              accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Numeric-only housing dataset (area, bedrooms, age → price)."""
    rng = np.random.default_rng(seed)
    area     = rng.uniform(500, 3000, n).round(0)
    bedrooms = rng.integers(1, 6, n)
    age      = rng.uniform(0, 50, n).round(1)
    price    = (area * 150 + bedrooms * 10_000 - age * 1_000
                + rng.standard_normal(n) * 10_000).round(-2)
    return pd.DataFrame({'area': area.astype(int), 'bedrooms': bedrooms,
                         'age': age, 'price': price.astype(int)})


def make_classification_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Student exam dataset: hours_studied + hours_sleep → passed (0/1)."""
    rng          = np.random.default_rng(seed)
    hours_studied = rng.uniform(0, 10, n).round(1)
    hours_sleep   = rng.uniform(4, 10, n).round(1)
    noise         = rng.standard_normal(n)
    score         = 1.5 * hours_studied + 0.5 * hours_sleep + noise
    passed        = (score > 9.0).astype(int)
    return pd.DataFrame({'hours_studied': hours_studied,
                         'hours_sleep':   hours_sleep,
                         'passed':        passed})


def cross_validate_model(model, X: pd.DataFrame, y: pd.Series,
                          cv: int = 5,
                          scoring: str = 'r2') -> dict:
    """K-fold cross-validation returning per-fold scores and summary stats."""
    kf     = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring=scoring)
    return {
        'scores':   scores,
        'mean':     round(float(scores.mean()), 4),
        'std':      round(float(scores.std()),  4),
        'min':      round(float(scores.min()),  4),
        'max':      round(float(scores.max()),  4),
        'cv_folds': cv,
        'scoring':  scoring,
    }


def overfitting_report(X: pd.DataFrame, y: pd.Series,
                        max_depths=range(1, 11),
                        test_size: float = 0.2,
                        random_state: int = 42) -> pd.DataFrame:
    """Train DecisionTreeRegressors at each depth; return train vs test R² table."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    records = []
    for depth in max_depths:
        m = DecisionTreeRegressor(max_depth=depth, random_state=42)
        m.fit(X_train, y_train)
        tr_r2 = float(r2_score(y_train, m.predict(X_train)))
        te_r2 = float(r2_score(y_test,  m.predict(X_test)))
        gap   = round(tr_r2 - te_r2, 4)
        records.append({
            'max_depth': depth,
            'train_r2':  round(tr_r2, 4),
            'test_r2':   round(te_r2, 4),
            'gap':       gap,
            'overfit':   bool(gap > 0.1),
        })
    return pd.DataFrame(records)

## Your Implementation

In [ ]:
def train_classifier(X_train: pd.DataFrame,
                     y_train: pd.Series,
                     max_iter: int = 1000) -> LogisticRegression:
    """
    Fit LogisticRegression for binary or multi-class classification.

    Args:
        X_train:  feature matrix (numeric, should be scaled)
        y_train:  target labels
        max_iter: solver iteration limit (default 1000 avoids ConvergenceWarning)
    Returns:
        Fitted LogisticRegression model
    """
    # TODO: model = LogisticRegression(random_state=42, max_iter=max_iter)
    # TODO: model.fit(X_train, y_train)
    # TODO: return model
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df_c = make_classification_data(200)
    X_c  = df_c.drop(columns=['passed'])
    y_c  = df_c['passed']
    X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
        X_c, y_c, test_size=0.2, random_state=42
    )
    scaler    = StandardScaler()
    X_tr_s    = pd.DataFrame(scaler.fit_transform(X_tr_c), columns=X_c.columns)
    X_te_s    = pd.DataFrame(scaler.transform(X_te_c),     columns=X_c.columns)

    # Check 1: defined, returns LogisticRegression
    try:
        assert 'train_classifier' in globals()
        clf = train_classifier(X_tr_s, y_tr_c)
        assert isinstance(clf, LogisticRegression), \
            f'expected LogisticRegression, got {type(clf).__name__}'
        passed += 1; print('\u2705 Check 1: train_classifier returns LogisticRegression')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: model is fitted (has coef_ attribute)
    try:
        assert hasattr(clf, 'coef_'), 'model must be fitted (has coef_)'
        passed += 1; print('\u2705 Check 2: model is fitted (has coef_)')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: coef_ shape matches n_features
    try:
        assert clf.coef_.shape[1] == X_c.shape[1], \
            f'coef_ n_features={clf.coef_.shape[1]}, expected {X_c.shape[1]}'
        passed += 1; print(f'\u2705 Check 3: coef_ shape {clf.coef_.shape} matches {X_c.shape[1]} features')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: accuracy on test set > 0.80
    try:
        y_pred = clf.predict(X_te_s)
        acc    = accuracy_score(y_te_c, y_pred)
        assert acc > 0.80, f'accuracy should be > 0.80, got {acc:.4f}'
        passed += 1; print(f'\u2705 Check 4: test accuracy={acc:.4f} > 0.80')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: model has predict_proba (LogisticRegression feature)
    try:
        assert hasattr(clf, 'predict_proba'), \
            'LogisticRegression should have predict_proba method'
        proba = clf.predict_proba(X_te_s)
        assert proba.shape == (len(X_te_s), 2), \
            f'predict_proba shape {proba.shape} should be ({len(X_te_s)}, 2)'
        passed += 1; print(f'\u2705 Check 5: predict_proba works, shape={proba.shape}')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def train_classifier(X_train: pd.DataFrame,
                     y_train: pd.Series,
                     max_iter: int = 1000) -> LogisticRegression:
    """Fit LogisticRegression and return the fitted model."""
    model = LogisticRegression(random_state=42, max_iter=max_iter)
    model.fit(X_train, y_train)
    return model
```

</details>